# Project Part 2 — Lossless Source Coding
## Audio Source (8-bit quantization)


In [1]:
import math, collections, heapq, wave, struct, os, subprocess, tempfile
import matplotlib.pyplot as plt

# Load and quantize audio (same as Part 1)
def read_wav(f):
    with wave.open(f,'rb') as wf:
        nc=wf.getnchannels(); sw=wf.getsampwidth()
        nf=wf.getnframes(); rb=wf.readframes(nf); fr=wf.getframerate()
    fmt='<'+str(nf*nc)+('h' if sw==2 else 'b')
    s=list(struct.unpack(fmt,rb))
    return (s[::2] if nc==2 else s), fr

def quantize(samples, n_bits):
    n_lvl=2**n_bits
    return [max(0,min(n_lvl-1,int((s+32768)/65535*(n_lvl-1)))) for s in samples]

raw_samples, framerate = read_wav('Audio.wav')
symbols = quantize(raw_samples, 8)  # 8-bit quantization
n = len(symbols)
counts = collections.Counter(symbols)
probs  = {x: c/n for x,c in counts.items()}

def entropy(p): return -sum(v*math.log2(v) for v in p.values() if v>0)
H = entropy(probs)
print(f'Total samples   : {n}')
print(f'Distinct levels : {len(probs)}')
print(f'H(X) 8-bit      : {H:.4f} bits/sample')

Total samples   : 5422080
Distinct levels : 116
H(X) 8-bit      : 6.1619 bits/sample


---
## Task 1 — Huffman Coding
### Q1a: Build Huffman code from empirical probabilities

In [2]:
def build_huffman(prob_dict):
    heap = [[p,i,sym,None,None] for i,(sym,p) in enumerate(prob_dict.items())]
    import heapq; heapq.heapify(heap)
    nid = len(prob_dict)
    while len(heap) > 1:
        l = heapq.heappop(heap); r = heapq.heappop(heap)
        heapq.heappush(heap, [l[0]+r[0], nid, None, l, r]); nid+=1
    return heap[0]

def get_codes(tree, prefix='', codes={}):
    if tree[2] is not None: codes[tree[2]] = prefix or '0'
    else: get_codes(tree[3],prefix+'0',codes); get_codes(tree[4],prefix+'1',codes)
    return codes

import heapq
tree  = build_huffman(probs)
codes = get_codes(tree, '', {})

E_l = sum(probs[s]*len(codes[s]) for s in probs)
print(f'E[l(X)] Huffman : {E_l:.4f} bits/sample')
print(f'H(X)            : {H:.4f} bits/sample')
print(f'Gap             : {E_l-H:.4f} bits')

E[l(X)] Huffman : 6.1899 bits/sample
H(X)            : 6.1619 bits/sample
Gap             : 0.0280 bits


### Q1b: Encode and verify lossless compression

In [3]:
sample = symbols[:5000]
bitstream = ''.join(codes[s] for s in sample)
avg_len = len(bitstream) / len(sample)

# Decode
rev = {v:k for k,v in codes.items()}
decoded, buf = [], ''
for bit in bitstream:
    buf += bit
    if buf in rev: decoded.append(rev[buf]); buf=''

print(f'Encoded 5000 samples -> {len(bitstream)} bits')
print(f'Avg bits/sample : {avg_len:.4f}')
print(f'H(X)            : {H:.4f}')
print(f'Lossless?       : {decoded == sample}')

Encoded 5000 samples -> 20000 bits
Avg bits/sample : 4.0000
H(X)            : 6.1619
Lossless?       : True


### Q1c: Compare across quantization levels

In [4]:
print('Huffman performance at different quantization resolutions:')
for bits in [4, 8, 16]:
    sym_b = quantize(raw_samples, bits)
    p_b = {x:c/len(sym_b) for x,c in collections.Counter(sym_b).items()}
    H_b = entropy(p_b)
    tree_b = build_huffman(p_b)
    codes_b = get_codes(tree_b,'',{})
    El_b = sum(p_b[s]*len(codes_b[s]) for s in p_b)
    print(f'  {bits}-bit: H={H_b:.4f}, E[l]={El_b:.4f}, gap={El_b-H_b:.4f}')

Huffman performance at different quantization resolutions:
  4-bit: H=2.1688, E[l]=2.2191, gap=0.0503
  8-bit: H=6.1619, E[l]=6.1899, gap=0.0280
  16-bit: H=14.0548, E[l]=14.0835, gap=0.0287


---
## Task 2 — Block Coding
### Q2a: Huffman on blocks of consecutive samples

In [5]:
def block_huffman_avg(sym_list, block_size):
    blocks = [tuple(sym_list[i:i+block_size]) for i in range(0,len(sym_list)-block_size+1,block_size)]
    n_b = len(blocks)
    bp = {b:c/n_b for b,c in collections.Counter(blocks).items()}
    tree = build_huffman(bp)
    codes_b = get_codes(tree,'',{})
    return sum(bp[b]*len(codes_b[b]) for b in bp) / block_size

print('Block Huffman (8-bit audio):')
results = []
for bs in [1, 2, 3]:
    avg = block_huffman_avg(symbols[:10000], bs)
    results.append((bs, avg))
    print(f'  Block {bs}: {avg:.4f} bits/sample')
print(f'  H(X)  : {H:.4f}')

Block Huffman (8-bit audio):
  Block 1: 1.0000 bits/sample
  Block 2: 0.5000 bits/sample
  Block 3: 0.3333 bits/sample
  H(X)  : 6.1619


---
## Task 3 — Arithmetic Coding
### Q3a: Memoryless arithmetic coding

In [6]:
def build_cum(p):
    alph = sorted(p.keys())
    cum = {}; c = 0.0
    for s in alph: cum[s]=(c, c+p[s]); c+=p[s]
    return cum

def arith_encode(seq, p):
    cum = build_cum(p)
    L,U,bits,pend = 0.0,1.0,[],0
    for s in seq:
        if s not in cum: continue
        lo,hi = cum[s]; w = U-L
        U=L+w*hi; L=L+w*lo
        while True:
            if U<=0.5:   bits+=[0]+[1]*pend; pend=0; L*=2;U*=2
            elif L>=0.5: bits+=[1]+[0]*pend; pend=0; L=(L-.5)*2;U=(U-.5)*2
            elif L>=.25 and U<=.75: pend+=1;L=(L-.25)*2;U=(U-.25)*2
            else: break
    pend+=1
    if L<.25: bits+=[0]+[1]*pend
    else:     bits+=[1]+[0]*pend
    return bits

sample_ac = symbols[:1000]
bits_ml = arith_encode(sample_ac, probs)
print(f'Arithmetic coding memoryless: {len(bits_ml)/len(sample_ac):.4f} bits/sample')
print(f'H(X): {H:.4f}')

Arithmetic coding memoryless: 3.8100 bits/sample
H(X): 6.1619


### Q3b: Markov model (k=1)

In [7]:
trans = collections.defaultdict(collections.Counter)
for i in range(len(symbols)-1): trans[symbols[i]][symbols[i+1]]+=1
trans_p = {ctx:{s:c/sum(cn.values()) for s,c in cn.items()} for ctx,cn in trans.items()}

def arith_encode_markov(seq, trans_p, init_p):
    L,U,bits,pend,prev = 0.0,1.0,[],0,None
    for s in seq:
        p_model = trans_p.get(prev, init_p) if prev is not None else init_p
        cum = build_cum(p_model)
        if s not in cum: prev=s; continue
        lo,hi=cum[s]; w=U-L; U=L+w*hi; L=L+w*lo
        while True:
            if U<=0.5:   bits+=[0]+[1]*pend;pend=0;L*=2;U*=2
            elif L>=0.5: bits+=[1]+[0]*pend;pend=0;L=(L-.5)*2;U=(U-.5)*2
            elif L>=.25 and U<=.75: pend+=1;L=(L-.25)*2;U=(U-.25)*2
            else: break
        prev=s
    pend+=1
    if L<.25: bits+=[0]+[1]*pend
    else:     bits+=[1]+[0]*pend
    return bits

bits_mk = arith_encode_markov(sample_ac, trans_p, probs)
print(f'Arithmetic Markov k=1 : {len(bits_mk)/len(sample_ac):.4f} bits/sample')
print(f'Memoryless            : {len(bits_ml)/len(sample_ac):.4f} bits/sample')
print(f'H(X)                  : {H:.4f} bits/sample')

Arithmetic Markov k=1 : 0.4520 bits/sample
Memoryless            : 3.8100 bits/sample
H(X)                  : 6.1619 bits/sample


---
## Task 4 — Summary and Comparison
### Q4a: Full results table

In [8]:
print('=== Compression Results — Audio (8-bit quantization) ===')
print(f'{"Method":<35} {"Bits/sample":>12}')
print('-'*50)
print(f'{"H(X) — entropy bound":<35} {H:>12.4f}')
print(f'{"Huffman (symbol)":<35} {E_l:>12.4f}')
for bs, avg in results:
    print(f'{f"Block Huffman (n={bs})":<35} {avg:>12.4f}')
print(f'{"Arithmetic (memoryless)":<35} {len(bits_ml)/len(sample_ac):>12.4f}')
print(f'{"Arithmetic (Markov k=1)":<35} {len(bits_mk)/len(sample_ac):>12.4f}')

=== Compression Results — Audio (8-bit quantization) ===
Method                               Bits/sample
--------------------------------------------------
H(X) — entropy bound                      6.1619
Huffman (symbol)                          6.1899
Block Huffman (n=1)                       1.0000
Block Huffman (n=2)                       0.5000
Block Huffman (n=3)                       0.3333
Arithmetic (memoryless)                   3.8100
Arithmetic (Markov k=1)                   0.4520


### Q4b: System compressors comparison

In [ ]:
# Save raw quantized audio as bytes and compress with system tools
raw_bytes = bytes(symbols)
with tempfile.NamedTemporaryFile(delete=False, suffix='.bin') as f:
    f.write(raw_bytes); tmp_path = f.name

original_bytes = os.path.getsize(tmp_path)
original_bps   = 8  # 1 byte = 8 bits per sample

for tool, ext in [('gzip','.gz'),('bzip2','.bz2'),('xz','.xz')]:
    try:
        subprocess.run([tool,'-k',tmp_path], capture_output=True)
        out = tmp_path + ext
        if os.path.exists(out):
            cb = os.path.getsize(out)
            bps = cb/original_bytes*8
            print(f'{tool:8s}: {bps:.4f} bits/sample (ratio={cb/original_bytes:.4f})')
            os.remove(out)
    except Exception as e:
        print(f'{tool}: not available')

os.remove(tmp_path)
print(f'\nFor reference: H(X) = {H:.4f} bits/sample')